# Cubic model: Gaussian-preparation dynamics

This notebook reproduces the manuscript dynamics figure for a Gaussian initial condition. It compares UCNA and LLA only and uses compact plot data by default. Set `USE_PRECOMPUTED_PLOT_DATA=False` and `RUN_SIMULATIONS=True` to generate replica files.


In [ ]:
from dataclasses import asdict, dataclass, replace
from functools import lru_cache
from pathlib import Path
import pickle

import matplotlib as mpl
import matplotlib.pyplot as plt
import numba as nb
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.integrate import cumulative_trapezoid, quad
from scipy.special import dawsn
from tqdm.auto import tqdm

plt.rcParams.update({
    'axes.labelsize': 11,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,    
})

from ucna_utils import (
    kl_divergence, replica_raw_kl_uncertainty, simulate_endpoints,
    simulate_snapshot_counts, solve_fokker_planck_banded, width_binned,
)

RUN_SIMULATIONS = False
RUN_ANALYSIS = True
FORCE_REANALYSIS = False
OUTPUT_DIR = Path('results/cubic_dynamics')
USE_PRECOMPUTED_PLOT_DATA = True
PLOT_DATA_DIR = Path('plot_data')


In [ ]:
@nb.njit
def cubic_drift(x):
    return -x**3

def cubic_drift_numpy(x):
    return -np.asarray(x)**3

def cubic_drift_prime(x):
    return -3.0*np.asarray(x)**2

def cubic_drift_second(x):
    return -6.0*np.asarray(x)

def cubic_ucna_density(x, tau, D):
    x = np.asarray(x)
    gamma = 1.0 + 3.0*tau*x**2
    return gamma*np.exp((-0.5*tau*x**6 - 0.25*x**4)/D)

@lru_cache(maxsize=None)
def cubic_ucna_standard_deviation(tau, D):
    # The density is even, hence its mean is exactly zero.
    normalization = 2.0*quad(
        lambda x: cubic_ucna_density(x, tau, D), 0.0, np.inf,
        epsabs=1e-11, epsrel=1e-10, limit=200,
    )[0]
    second_moment = 2.0*quad(
        lambda x: x*x*cubic_ucna_density(x, tau, D), 0.0, np.inf,
        epsabs=1e-11, epsrel=1e-10, limit=200,
    )[0]/normalization
    if not np.isfinite(second_moment) or second_moment <= 0:
        raise ValueError('UCNA variance is nonpositive or nonfinite')
    return float(np.sqrt(second_moment))


## Production configurations and simulation


In [ ]:
@dataclass(frozen=True)
class DynamicsExperiment:
    tau: float
    D: float
    protocol: str = 'gaussian'
    dt: float = 0.01
    pde_dt: float = 0.005
    snapshot_dt: float = 0.05
    n_replicas: int = 10
    samples_per_replica: int = 10_000
    n_bins: int = 200
    pde_points: int = 401
    tail_tolerance: float = 1e-8
    base_seed: int = 20260814

    @property
    def quench_factor(self):
        return {'gaussian': 1.0, 'up_quench': 0.5, 'down_quench': 2.0}[self.protocol]

    @property
    def D0(self):
        return self.quench_factor*self.D

    @property
    def duration(self):
        return 10.0*np.sqrt(self.tau)/self.D**0.25

    @property
    def burn_duration(self):
        if self.protocol == 'gaussian':
            return 0.0
        return 10.0*np.sqrt(self.tau)/self.D0**0.25

    @property
    def n_steps(self):
        return int(np.ceil(self.duration/self.dt))

    @property
    def burn_steps(self):
        return int(np.ceil(self.burn_duration/self.dt))

    @property
    def snapshot_stride(self):
        stride = int(round(self.snapshot_dt/self.dt))
        if stride < 1 or not np.isclose(stride*self.dt, self.snapshot_dt):
            raise ValueError('snapshot_dt must be an integer multiple of dt')
        return stride

    @property
    def snapshot_steps(self):
        steps = np.arange(0, self.n_steps+1, self.snapshot_stride, dtype=np.int64)
        if steps[-1] != self.n_steps:
            steps = np.append(steps, self.n_steps)
        return steps

    @property
    def gaussian_sigma(self):
        return cubic_ucna_standard_deviation(self.tau, self.D0)

    @property
    def half_width(self):
        widths = [6.0*self.gaussian_sigma]
        for current_D in {self.D, self.D0}:
            widths.append(width_binned(
                lambda x, current_D=current_D: cubic_ucna_density(x, self.tau, current_D),
                tail_tolerance=self.tail_tolerance,
            ))
        return float(max(widths))

TAUS = [1.0, 2.0, 4.0, 8.0, 16.0]
DS = [0.1, 0.3, 1.0, 3.0, 10.0]
PROTOCOLS = ['gaussian']#['gaussian', 'up_quench', 'down_quench']
EXPERIMENTS = [
    DynamicsExperiment(
        tau=tau, D=D, protocol=protocol,
        n_replicas=10,
        samples_per_replica=100000
    )
    for protocol in PROTOCOLS for tau in TAUS for D in DS
]

cost = pd.DataFrame(asdict(config) | {
    'duration': config.duration, 'burn_duration': config.burn_duration,
    'trajectory_steps': config.n_replicas*config.samples_per_replica
                        *(config.n_steps+config.burn_steps),
} for config in EXPERIMENTS)
display(cost.groupby('protocol').agg(
    configurations=('D', 'size'), trajectory_steps=('trajectory_steps', 'sum')
))


In [ ]:
def replica_seed(config, replica, stage):
    stage_code = {'initial': 0x101, 'burn': 0x202, 'transient': 0x303}[stage]
    protocol_code = {'gaussian': 1, 'up_quench': 2, 'down_quench': 3}[config.protocol]
    return int(np.random.SeedSequence([
        config.base_seed, protocol_code, replica, stage_code,
        int(round(config.tau*1e6)), int(round(config.D*1e6)),
    ]).generate_state(1, dtype=np.uint32)[0])

def supplemental_replica_seed(config, replica, stage, previous_sample_count):
    stage_code = {'initial': 0x101, 'burn': 0x202, 'transient': 0x303}[stage]
    protocol_code = {'gaussian': 1, 'up_quench': 2, 'down_quench': 3}[config.protocol]
    return int(np.random.SeedSequence([
        config.base_seed, protocol_code, replica, stage_code,
        int(round(config.tau*1e6)), int(round(config.D*1e6)),
        0xA5A5A5A5, int(previous_sample_count),
    ]).generate_state(1, dtype=np.uint32)[0])

def result_directory(config):
    root = OUTPUT_DIR if config.protocol == 'gaussian' else OUTPUT_DIR/'reset_eta'
    return root/config.protocol/f'tau_{config.tau:g}_D_{config.D:g}'

def simulate_replica_chunk(config, replica, n_samples, seeds):
    edges = np.linspace(-config.half_width, config.half_width, config.n_bins+1)
    initial_x = None
    if config.protocol != 'gaussian':
        rng = np.random.default_rng(seeds['initial'])
        trial_x = rng.normal(0.0, config.gaussian_sigma, n_samples)
        initial_x, _ = simulate_endpoints(
            cubic_drift, config.D0, config.tau, config.dt, config.burn_steps,
            n_samples, seeds['burn'], initial_x=trial_x,
        )
    counts, outside, final_x, final_eta = simulate_snapshot_counts(
        cubic_drift, config.D, config.tau, config.dt, config.n_steps,
        config.snapshot_steps, edges, n_samples, seeds['transient'],
        initial_mean=0.0, initial_std=config.gaussian_sigma,
        # A new stationary OU value at the target D is drawn for every chunk.
        initial_x=initial_x, initial_eta=None,
    )
    return counts, outside, final_x, final_eta

def simulate_replica(config, replica):
    stages = ['initial', 'burn', 'transient']
    seeds = {stage: replica_seed(config, replica, stage) for stage in stages}
    counts, outside, final_x, final_eta = simulate_replica_chunk(
        config, replica, config.samples_per_replica, seeds,
    )
    return {
        'schema_version': 2,
        'config': asdict(config), 'replica': replica,
        'edges': np.linspace(-config.half_width, config.half_width, config.n_bins+1),
        'snapshot_steps': config.snapshot_steps,
        'times': config.snapshot_steps*config.dt,
        'counts': counts, 'outside': outside,
        'final_x': final_x, 'final_eta': final_eta,
        'quench_noise_initialization': (
            'not_applicable' if config.protocol == 'gaussian'
            else 'independent_target_stationary'
        ),
        'seeds': seeds,
        'seed_chunks': {stage: [seeds[stage]] for stage in stages},
        'chunk_sizes': [config.samples_per_replica],
    }

def extend_replica(saved, target_config):
    old = dict(saved['config'])
    target = asdict(target_config)
    sample_fields = {'n_replicas', 'samples_per_replica'}
    analysis_fields = {'pde_points', 'pde_dt'}
    changed_fixed = {
        key: (old.get(key), target[key]) for key in target
        if key not in sample_fields | analysis_fields
        and old.get(key) != target[key]
    }
    if changed_fixed:
        raise ValueError(f'Non-sample configuration changes cannot be appended: {changed_fixed}')
    if target['n_replicas'] < old['n_replicas']:
        raise ValueError('n_replicas cannot be reduced in existing dynamics results')
    old_samples = int(old['samples_per_replica'])
    target_samples = int(target['samples_per_replica'])
    if target_samples < old_samples:
        raise ValueError('samples_per_replica cannot be reduced in existing dynamics results')
    if len(saved['final_x']) != old_samples or len(saved['final_eta']) != old_samples:
        raise ValueError('Saved endpoint dimensions disagree with the saved configuration')

    updated = dict(saved)
    updated['config'] = target
    updated['schema_version'] = 2
    stages = ['initial', 'burn', 'transient']
    old_seed_chunks = saved.get('seed_chunks')
    if old_seed_chunks is None:
        old_seed_chunks = {
            stage: [int(saved['seeds'][stage])] for stage in stages
        }
    updated['seed_chunks'] = old_seed_chunks
    updated['chunk_sizes'] = list(map(
        int, saved.get('chunk_sizes', [old_samples])
    ))
    if target_samples == old_samples:
        return updated

    replica = int(saved['replica'])
    seeds = {
        stage: supplemental_replica_seed(
            target_config, replica, stage, old_samples
        ) for stage in stages
    }
    n_new = target_samples-old_samples
    counts, outside, final_x, final_eta = simulate_replica_chunk(
        target_config, replica, n_new, seeds,
    )
    if np.shape(counts) != np.shape(saved['counts']):
        raise ValueError('Saved snapshot histogram shape differs from the target configuration')
    if not np.allclose(
        saved['edges'],
        np.linspace(-target_config.half_width, target_config.half_width,
                    target_config.n_bins+1),
    ):
        raise ValueError('Saved histogram edges differ from the target configuration')

    updated.update(
        counts=np.asarray(saved['counts'])+counts,
        outside=np.asarray(saved['outside'])+outside,
        final_x=np.concatenate([saved['final_x'], final_x]),
        final_eta=np.concatenate([saved['final_eta'], final_eta]),
        seed_chunks={
            stage: list(map(int, old_seed_chunks[stage]))+[int(seeds[stage])]
            for stage in stages
        },
        chunk_sizes=updated['chunk_sizes']+[n_new],
    )
    return updated

def save_replica(result, filename):
    temporary = filename.with_suffix('.pkl.tmp')
    with temporary.open('wb') as handle:
        pickle.dump(result, handle)
    temporary.replace(filename)

def run_missing_replicas(configs):
    for config in tqdm(configs, desc='dynamics configurations'):
        directory = result_directory(config)
        directory.mkdir(parents=True, exist_ok=True)
        actions = []
        reusable = 0
        for replica in range(config.n_replicas):
            filename = directory/f'replica_{replica:03d}.pkl'
            if not filename.exists():
                actions.append(('new', replica, filename))
                continue
            with filename.open('rb') as handle:
                saved = pickle.load(handle)
            old, target = saved['config'], asdict(config)
            sample_fields = {'n_replicas', 'samples_per_replica'}
            analysis_fields = {'pde_points', 'pde_dt'}
            changed_fixed = {
                key for key in target
                if key not in sample_fields | analysis_fields
                and old.get(key) != target[key]
            }
            decreases = (
                target['n_replicas'] < old['n_replicas']
                or target['samples_per_replica'] < old['samples_per_replica']
            )
            if changed_fixed or decreases:
                raise ValueError(
                    f'{filename} differs in non-appendable settings: '
                    f'fields={sorted(changed_fixed)}, decrease={decreases}'
                )
            if target['samples_per_replica'] > old['samples_per_replica']:
                actions.append(('extend', replica, filename))
            elif target['n_replicas'] != old['n_replicas']:
                actions.append(('metadata', replica, filename))
            else:
                reusable += 1

        counts = {kind: sum(item[0] == kind for item in actions)
                  for kind in ['new', 'extend', 'metadata']}
        print(
            f'{config.protocol}: tau={config.tau:g}, D={config.D:g}: '
            f'{reusable} reusable, {counts["extend"]} extendable, '
            f'{counts["new"]} new, {counts["metadata"]} metadata-only'
        )
        for action, replica, filename in tqdm(actions, desc='replicas', leave=False):
            if action == 'new':
                result = simulate_replica(config, replica)
            else:
                with filename.open('rb') as handle:
                    saved = pickle.load(handle)
                result = extend_replica(saved, config)
            save_replica(result, filename)

def load_replicas(config):
    directory = result_directory(config)
    loaded = []
    for replica in range(config.n_replicas):
        filename = directory/f'replica_{replica:03d}.pkl'
        if not filename.exists():
            raise FileNotFoundError(f'missing {filename}')
        with filename.open('rb') as handle:
            result = pickle.load(handle)
        simulation_fields = set(asdict(config)) - {'pde_points', 'pde_dt'}
        if any(result['config'].get(key) != asdict(config)[key]
               for key in simulation_fields):
            raise ValueError(f'configuration mismatch in {filename}')
        loaded.append(result)
    return loaded


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    print('Using packaged dynamics plot data; replicas are not loaded.')
else:
    if RUN_SIMULATIONS:
        run_missing_replicas(EXPERIMENTS)


## Effective Fokker--Planck equations and analysis


In [ ]:
def drift_ucna(x, tau, D):
    gamma = 1.0 - tau*cubic_drift_prime(x)
    return cubic_drift_numpy(x)/gamma + D*tau*cubic_drift_second(x)/gamma**3

def diffusion_ucna(x, time, tau, D):
    return D/(1.0-tau*cubic_drift_prime(x))**2

def drift_lla(x, tau, D):
    return cubic_drift_numpy(x)

def diffusion_lla(x, time, tau, D):
    return D/(1.0-tau*cubic_drift_prime(x))

def drift_cbfpe(x, tau, D):
    return cubic_drift_numpy(x)

def diffusion_cbfpe(x, time, tau, D):
    x = np.asarray(x, dtype=float)
    r = tau*x*x
    result = np.empty_like(x)
    small = r < 1e-10
    result[small] = D*(-np.expm1(-time/tau))
    mask = ~small
    if np.any(mask):
        x2 = x[mask]**2
        rm = r[mask]
        cutoff = 1.0/(2.0*x2)
        horizon = np.minimum(time, cutoff)
        q = np.sqrt(np.maximum(1.0-2.0*x2*horizon, 0.0))
        z = 1.0/np.sqrt(2.0*rm)
        stationary = 1.0-3.0*rm+3.0*np.sqrt(2.0)*rm**1.5*dawsn(z)
        remainder = np.exp(-horizon/tau)*(
            q**3-3.0*rm*q+3.0*np.sqrt(2.0)*rm**1.5*dawsn(z*q)
        )
        result[mask] = D*(stationary-remainder)
    if np.any(~np.isfinite(result)) or np.any(result < -1e-12):
        raise FloatingPointError('invalid time-dependent cBFPE diffusion')
    return np.maximum(result, 0.0)

APPROXIMATIONS = {
    'UCNA': (drift_ucna, diffusion_ucna),
    'LLA': (drift_lla, diffusion_lla),
}


In [ ]:
def density_to_bin_probabilities(density, grid, edges):
    density = np.maximum(np.asarray(density, dtype=float), 0.0)
    n_bins = len(edges)-1
    intervals_per_bin, remainder = divmod(len(grid)-1, n_bins)
    aligned = (
        remainder == 0 and intervals_per_bin > 0
        and np.allclose(grid[::intervals_per_bin], edges)
    )
    if not aligned:
        raise ValueError('PDE grid must align with histogram edges')
    # Integrate locally, but do every bin at once. Unlike differences of a
    # global CDF, summing the trapezoids within each bin retains tail mass.
    interval_mass = 0.5*(density[:-1]+density[1:])*np.diff(grid)
    weights = interval_mass.reshape(n_bins, intervals_per_bin).sum(axis=1)
    if weights.sum() <= 0 or np.any(~np.isfinite(weights)):
        raise ValueError('PDE density produced invalid bin probabilities')
    return weights/weights.sum()

def transient_metrics(values, times, tail_fraction=0.2, smooth_fraction=0.03):
    values = np.asarray(values, dtype=float)
    times = np.asarray(times, dtype=float)
    if np.any(~np.isfinite(values)) or np.any(values < -1e-12):
        raise ValueError('transient KL must be finite and nonnegative')
    values = np.maximum(values, 0.0)
    n_tail = max(3, int(np.ceil(tail_fraction*len(values))))
    plateau = float(np.median(values[-n_tail:]))
    window = max(3, int(np.ceil(smooth_fraction*len(values))))
    window += 1-window%2
    smooth = pd.Series(values).rolling(window, center=True, min_periods=1).median().to_numpy()
    excess = np.maximum(smooth-plateau, 0.0)
    peak_index = int(np.argmax(excess))
    return {
        'KL_plateau': plateau, 'KL_peak': float(smooth[peak_index]),
        'KL_peak_excess': float(excess[peak_index]),
        't_peak': float(times[peak_index]),
        'KL_excess_area': float(np.trapezoid(np.maximum(values-plateau, 0), times)),
    }

def analyze_experiment(config):
    replicas = load_replicas(config)
    counts = np.stack([result['counts'] for result in replicas])
    outside = np.stack([result['outside'] for result in replicas])
    edges = replicas[0]['edges']
    times = replicas[0]['times']
    pooled = counts.sum(axis=0)
    total_samples = config.n_replicas*config.samples_per_replica
    grid = np.linspace(edges[0], edges[-1], config.pde_points)
    dx = grid[1]-grid[0]
    centers = 0.5*(edges[:-1]+edges[1:])
    initial_density = np.interp(grid, centers, pooled[0], left=0.0, right=0.0)
    initial_density /= initial_density.sum()*dx
    rows, trajectories = [], {}

    for name, (drift, diffusion) in APPROXIMATIONS.items():
        history, pde_times = solve_fokker_planck_banded(
            grid, config.tau, config.D, drift, diffusion, initial_density,
            dt=config.pde_dt, tmax=config.n_steps*config.dt,
        )
        position = times/config.pde_dt
        lower = np.floor(position + 1e-12).astype(int)
        upper = np.minimum(lower+1, len(history)-1)
        fraction = np.clip(position-lower, 0.0, 1.0)
        sampled_history = (
            (1.0-fraction[:, None])*history[lower]
            + fraction[:, None]*history[upper]
        )
        theory = np.stack([
            density_to_bin_probabilities(density, grid, edges)
            for density in sampled_history
        ])
        invalid = (theory <= 0.0) & (pooled > 0)
        if np.any(invalid):
            time_index, bin_index = np.argwhere(invalid)[0]
            reason = (
                f'nonpositive theoretical bin probability at t={times[time_index]:.6g}, '
                f'bin={bin_index}, observed_count={pooled[time_index, bin_index]}'
            )
            print(
                f'FLAGGED {config.protocol}, tau={config.tau:g}, D={config.D:g}, '
                f'{name}: {reason}'
            )
            rows.append({
                'protocol': config.protocol, 'tau': config.tau, 'D': config.D,
                'D0': config.D0, 'Approximation': name,
                'KL_plateau': np.nan, 'KL_peak': np.nan,
                'KL_peak_excess': np.nan, 't_peak': np.nan,
                'KL_excess_area': np.nan,
                'KL_peak_excess_uncertainty': np.nan,
                'max_outside_fraction': float(
                    outside.sum(axis=(0, 2)).max()/total_samples
                ),
                'analysis_valid': False, 'failure_reason': reason,
            })
            trajectories[name] = {
                'times': times, 'theory_bin_probabilities': theory,
                'analysis_valid': False, 'failure_reason': reason,
            }
            continue
        pooled_kl = np.empty(len(times))
        uncertainty = np.empty(len(times))
        raw_by_replica = np.empty((config.n_replicas, len(times)))
        for time_index in range(len(times)):
            pooled_kl[time_index] = (
                total_samples/(config.n_bins-1)
                * kl_divergence(pooled[time_index], theory[time_index])
            )
            _, _, uncertainty[time_index] = replica_raw_kl_uncertainty(
                counts[:, time_index], theory[time_index], total_samples
            )
            for replica in range(config.n_replicas):
                raw_by_replica[replica, time_index] = kl_divergence(
                    counts[replica, time_index], theory[time_index]
                )
        metrics = transient_metrics(pooled_kl, times)
        raw_peak_excess = np.array([
            transient_metrics(raw_values, times)['KL_peak_excess']
            for raw_values in raw_by_replica
        ])
        metrics['KL_peak_excess_uncertainty'] = (
            total_samples/(config.n_bins-1)
            * raw_peak_excess.std(ddof=1)/np.sqrt(config.n_replicas)
        )
        rows.append({
            'protocol': config.protocol, 'tau': config.tau, 'D': config.D,
            'D0': config.D0, 'Approximation': name, **metrics,
            'max_outside_fraction': float(outside.sum(axis=(0, 2)).max()/total_samples),
            'analysis_valid': True, 'failure_reason': '',
        })
        trajectories[name] = {
            'times': times, 'KL': pooled_kl, 'KL_uncertainty': uncertainty,
            'theory_bin_probabilities': theory,
        }
    return pd.DataFrame(rows), trajectories


ANALYSIS_CACHE_VERSION = 1
ANALYSIS_CACHE_FILE = OUTPUT_DIR/'analysis_cache.pkl'

def analysis_signature(configs):
    """Describe the requested analysis and the exact saved replica files."""
    files = []
    for config in configs:
        directory = result_directory(config)
        for replica in range(config.n_replicas):
            filename = directory/f'replica_{replica:03d}.pkl'
            if not filename.exists():
                return None
            stat = filename.stat()
            files.append((str(filename), stat.st_size, stat.st_mtime_ns))
    return {
        'version': ANALYSIS_CACHE_VERSION,
        'configs': [asdict(config) for config in configs],
        'approximations': tuple(APPROXIMATIONS),
        'replica_files': files,
    }

def load_analysis_cache(configs):
    if FORCE_REANALYSIS or not ANALYSIS_CACHE_FILE.exists():
        return None
    signature = analysis_signature(configs)
    if signature is None:
        print('Analysis cache ignored: one or more replica files are missing.')
        return None
    with ANALYSIS_CACHE_FILE.open('rb') as handle:
        cached = pickle.load(handle)
    if cached.get('signature') != signature:
        print('Analysis cache ignored: configurations or simulation files changed.')
        return None
    print(f'Loaded analysis from {ANALYSIS_CACHE_FILE}')
    return cached['dynamics_summary'], cached['transient_results']


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    analysis_was_recomputed = False
    dynamics_summary = pd.read_csv(PLOT_DATA_DIR / 'cubic_dynamics.csv')
else:
    analysis_was_recomputed = False
    if RUN_ANALYSIS:
        cached_analysis = load_analysis_cache(EXPERIMENTS)
        if cached_analysis is None:
            summary_parts = []
            transient_results = {}
            for config in tqdm(EXPERIMENTS, desc='analyzing dynamics'):
                summary, trajectories = analyze_experiment(config)
                summary_parts.append(summary)
                transient_results[(config.protocol, config.tau, config.D)] = trajectories
            dynamics_summary = pd.concat(summary_parts, ignore_index=True)
            dynamics_summary['tau2D'] = dynamics_summary.tau**2*dynamics_summary.D
            analysis_was_recomputed = True
        else:
            dynamics_summary, transient_results = cached_analysis

        display(dynamics_summary)
        flagged_analyses = dynamics_summary.loc[
            ~dynamics_summary.analysis_valid,
            ['protocol', 'tau', 'D', 'Approximation', 'failure_reason'],
        ].reset_index(drop=True)
        if len(flagged_analyses):
            print(f'Flagged {len(flagged_analyses)} approximation/configuration pairs:')
            display(flagged_analyses)
        else:
            print('No analysis failures were flagged.')
    else:
        print('Analysis disabled. Set RUN_ANALYSIS=True after all requested replicas exist.')


In [ ]:
if USE_PRECOMPUTED_PLOT_DATA:
    print('Using packaged dynamics summary; no analysis cache is needed.')
else:
    if RUN_ANALYSIS and analysis_was_recomputed:
        signature = analysis_signature(EXPERIMENTS)
        if signature is None:
            raise FileNotFoundError('Cannot cache analysis while replica files are missing')
        ANALYSIS_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
        temporary = ANALYSIS_CACHE_FILE.with_suffix('.pkl.tmp')
        with temporary.open('wb') as handle:
            pickle.dump({
                'signature': signature,
                'dynamics_summary': dynamics_summary,
                'transient_results': transient_results,
            }, handle, protocol=pickle.HIGHEST_PROTOCOL)
        temporary.replace(ANALYSIS_CACHE_FILE)
        print(f'Saved analysis to {ANALYSIS_CACHE_FILE}')
    elif RUN_ANALYSIS:
        print('Analysis cache is current; nothing to save.')


## Dynamics figure


In [ ]:
if RUN_ANALYSIS:
    paper_data = dynamics_summary.query(
        "protocol == 'gaussian' and Approximation in ['UCNA', 'LLA'] "
        "and analysis_valid"
    ).copy()
    paper_taus = sorted(paper_data.tau.unique())
    n_tau = len(paper_taus)
    tau_colors = sns.color_palette(
        'viridis', n_colors=n_tau + 2*max(n_tau-1, 0)
    )[::3] if n_tau > 1 else sns.color_palette('viridis', n_colors=1)
    tau_cmap = mpl.colors.ListedColormap(tau_colors)
    tau_norm = mpl.colors.BoundaryNorm(
        np.arange(n_tau+1)-0.5, ncolors=n_tau
    )
    paper_styles = {
        'UCNA': dict(marker='o', linestyle='-'),
        'LLA': dict(marker='o', linestyle='--', markerfacecolor='white'),
    }

    fig, axis = plt.subplots(figsize=(4.5, 3.5))
    for tau_index, (tau, tau_data) in enumerate(paper_data.groupby('tau')):
        color = tau_colors[paper_taus.index(tau)]
        for name in ['UCNA', 'LLA']:
            curve = tau_data.query('Approximation == @name').sort_values('tau2D')
            if curve.empty:
                continue
            values = curve.KL_peak_excess.to_numpy()
            uncertainty = curve.KL_peak_excess_uncertainty.to_numpy()
            lower = np.maximum(values-uncertainty, np.finfo(float).tiny)
            upper = values+uncertainty
            axis.fill_between(
                curve.tau2D, lower, upper, color=color, alpha=0.25,
                linewidth=0,
            )
            axis.plot(
                curve.tau2D, values,
                color=color,  markeredgecolor=color,
                markersize=4.5, markeredgewidth=1.0, linewidth=1.1,
                **paper_styles[name],
            )

    scalar_map = mpl.cm.ScalarMappable(cmap=tau_cmap, norm=tau_norm)
    scalar_map.set_array([])
    colorbar = fig.colorbar(
        scalar_map, ax=axis, location='top', shrink=1, pad=0.02,
        ticks=range(n_tau),
    )
    colorbar.ax.set_xticklabels(
        [rf'$\tau={paper_taus[0]:g}$']
        + [rf'${tau:g}$' for tau in paper_taus[1:]]
    )
    approximation_handles = [
        mpl.lines.Line2D(
            [], [], color='black', marker=paper_styles['UCNA']['marker'],
            linestyle=paper_styles['UCNA']['linestyle'],
            markeredgecolor='black', markersize=4.5, linewidth=1.1, label='UCNA',
        ), 
        mpl.lines.Line2D(
                    [], [], color='black', marker=paper_styles['LLA']['marker'],
                    linestyle=paper_styles['LLA']['linestyle'], markerfacecolor = 'white',
                    markeredgecolor='black', markersize=4.5, linewidth=1.1, label='LLA',
                )
    ]
    axis.legend(handles=approximation_handles)
    axis.set(
        xscale='log', yscale='log', xlabel=r'$\mathbf{D\tau^2}$',
        ylabel=r'$\mathbf{d_{KL}^{max}-d_{KL}^{\infty}}$',
    )
    #sns.despine(ax=axis)
    fig.tight_layout()
    plt.show()
else:
    print('Run the main analysis first; no new simulation is required.')
